In [ ]:
%matplotlib widget

In [ ]:
import hyperspy.api as hs
import pyxem as pxm
import matplotlib.pyplot as plt
import numpy as np

from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import root, least_squares

from scipy.ndimage import shift, distance_transform_edt
from skimage.transform import AffineTransform, warp

from pathlib import Path

import importlib
import get_transforms
import get_masks
importlib.reload(get_transforms)
importlib.reload(get_masks)

In [ ]:
#funksjon som laster inn zspy-fil og summerer diffrkasjonsmønsteret --> brightfield-bilde
def process_stem(file_path):
    s = hs.load(file_path, lazy=True)
    s_bs = s.sum(axis=(-1, -2))
    s_bs.compute()
    return s_bs, s

#bruker funksjonen på alle zspy-filene fra 002 - 023 = 22 filer
data_folder = Path("data")
files = sorted(data_folder.glob("*.zspy"))
original_images = {}
s_dir = {}

for file in files:
    s_bs, s = process_stem(file)
    original_images[file.stem] = s_bs
    s_dir[file.stem] = s

#Sjekker om filene er prosessert: 
print(f"Processed {len(original_images)} files")

In [ ]:
registered_images, transformations, shifts = get_transforms.main(original_images)

In [ ]:
# Mask is not really dependent on transformation, 002 is just normalized
# Center is (row, column)
masks, center, radii = get_masks.main(original_images, registered_images)

In [ ]:
import numpy as np

from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import distance_transform_edt
from scipy.optimize import least_squares


def find_root(
    data,
    mask,
    previous_root=None,
    previous_previous_root=None,
    expected_winding=None,
    max_normalized_residual=0.05,
    max_step=6.0,
    max_prediction_error=4.0,
    winding_radius=2.0,
    winding_tolerance=0.25,
    minimum_edge_distance=1.0,
    duplicate_tolerance=0.25,
    return_diagnostics=False,
):
    """
    Find and track one vortex root in a complex vector field.

    Important
    ---------
    This function does not itself enforce irreversible disappearance.
    The result-generation loop below does that by no longer calling
    find_root after a vortex has disappeared.

    Parameters
    ----------
    data : 2D complex ndarray
        Complex DPC field.

    mask : 2D bool ndarray
        Circular mask for this vortex.

    previous_root : array-like or None
        Accepted root in the preceding image, in (row, column) order.

    previous_previous_root : array-like or None
        Accepted root two images earlier. When available, these two
        preceding roots define a constant-velocity prediction.

    expected_winding : +1, -1, or None
        Winding sign measured while the root was safely inside the mask.
        A measurable candidate with the opposite sign is rejected.

    max_normalized_residual : float
        Maximum |D(root)| / median(|D| in mask).

    max_step : float
        Hard maximum movement from the preceding root, in pixels.

    max_prediction_error : float
        Hard maximum distance from the constant-velocity prediction.
        Applied only when two preceding roots are available.

    winding_radius : float
        Radius used for winding measurement.

    winding_tolerance : float
        Unit winding is accepted when abs(abs(winding) - 1) is no more
        than this value.

    minimum_edge_distance : float
        Root must be at least this far inside the mask.

    duplicate_tolerance : float
        Fitted roots closer than this are treated as duplicates.

    return_diagnostics : bool
        If True, return (root, diagnostics).

    Returns
    -------
    root : ndarray, shape (2,)
        Root in (row, column) order, or [nan, nan].

    diagnostics : dict
        Returned when return_diagnostics=True.
    """

    data = np.asarray(data)
    mask = np.asarray(mask, dtype=bool)

    no_root = np.array([np.nan, np.nan], dtype=float)

    diagnostics = {
        "accepted": False,
        "reason": None,
        "root": no_root.copy(),
        "candidate_cell_count": 0,
        "fitted_candidate_count": 0,
        "valid_candidate_count": 0,
        "normalized_residual": np.nan,
        "absolute_residual": np.nan,
        "winding": np.nan,
        "winding_measurable": False,
        "distance_to_edge": np.nan,
        "distance_from_previous": np.nan,
        "prediction_error": np.nan,
        "predicted_root": no_root.copy(),
        "expected_winding": expected_winding,
        "score": np.nan,
        "candidates": [],
    }

    def finish(root, reason):
        root = np.asarray(root, dtype=float)
        diagnostics["reason"] = reason
        diagnostics["root"] = root.copy()

        if return_diagnostics:
            return root, diagnostics

        return root

    def is_valid_root(root):
        if root is None:
            return False

        root = np.asarray(root, dtype=float)

        return (
            root.shape == (2,)
            and np.all(np.isfinite(root))
        )

    if data.ndim != 2:
        raise ValueError("data must be two-dimensional")

    if mask.shape != data.shape:
        raise ValueError(
            "mask and data must have the same shape"
        )

    finite_pixels = (
        np.isfinite(data.real)
        & np.isfinite(data.imag)
    )

    valid_mask = mask & finite_pixels

    if not np.any(valid_mask):
        return finish(no_root, "empty_or_invalid_mask")

    field_magnitudes = np.abs(data[valid_mask])
    field_magnitudes = field_magnitudes[
        np.isfinite(field_magnitudes)
    ]

    if field_magnitudes.size == 0:
        return finish(no_root, "no_finite_field_values")

    field_scale = float(
        np.median(field_magnitudes)
    )

    if not np.isfinite(field_scale) or field_scale <= 0:
        return finish(no_root, "invalid_field_scale")

    n_rows, n_cols = data.shape

    rows = np.arange(n_rows, dtype=float)
    cols = np.arange(n_cols, dtype=float)

    vector_field = np.stack(
        (data.real, data.imag),
        axis=-1,
    )

    field_interpolator = RegularGridInterpolator(
        (rows, cols),
        vector_field,
        method="linear",
        bounds_error=False,
        fill_value=np.nan,
    )

    real_interpolator = RegularGridInterpolator(
        (rows, cols),
        data.real,
        method="linear",
        bounds_error=False,
        fill_value=np.nan,
    )

    imag_interpolator = RegularGridInterpolator(
        (rows, cols),
        data.imag,
        method="linear",
        bounds_error=False,
        fill_value=np.nan,
    )

    def evaluate_field(position):
        value = field_interpolator(
            np.asarray(position, dtype=float)
        )

        return np.asarray(
            value,
            dtype=float,
        ).reshape(-1)

    def calculate_winding(root):
        theta = np.linspace(
            0.0,
            2.0 * np.pi,
            32,
            endpoint=False,
        )

        points = np.column_stack((
            root[0]
            + winding_radius * np.sin(theta),

            root[1]
            + winding_radius * np.cos(theta),
        ))

        inside_image = (
            np.all(points[:, 0] >= 0)
            and np.all(points[:, 0] <= n_rows - 1)
            and np.all(points[:, 1] >= 0)
            and np.all(points[:, 1] <= n_cols - 1)
        )

        if not inside_image:
            return np.nan, False

        sample_rows = np.clip(
            np.rint(points[:, 0]).astype(int),
            0,
            n_rows - 1,
        )

        sample_cols = np.clip(
            np.rint(points[:, 1]).astype(int),
            0,
            n_cols - 1,
        )

        if not np.all(
            mask[sample_rows, sample_cols]
        ):
            return np.nan, False

        real_values = np.asarray(
            real_interpolator(points)
        ).reshape(-1)

        imag_values = np.asarray(
            imag_interpolator(points)
        ).reshape(-1)

        complex_values = (
            real_values + 1j * imag_values
        )

        if not np.all(np.isfinite(complex_values)):
            return np.nan, False

        phases = np.angle(complex_values)

        phase_steps = np.angle(
            np.exp(
                1j * (
                    np.roll(phases, -1)
                    - phases
                )
            )
        )

        winding = float(
            np.sum(phase_steps)
            / (2.0 * np.pi)
        )

        return winding, True

    # Predict the next root using constant velocity.
    if (
        is_valid_root(previous_root)
        and is_valid_root(previous_previous_root)
    ):
        previous_root_array = np.asarray(
            previous_root,
            dtype=float,
        )

        previous_previous_array = np.asarray(
            previous_previous_root,
            dtype=float,
        )

        velocity = (
            previous_root_array
            - previous_previous_array
        )

        predicted_root = (
            previous_root_array + velocity
        )

    elif is_valid_root(previous_root):
        previous_root_array = np.asarray(
            previous_root,
            dtype=float,
        )

        predicted_root = (
            previous_root_array.copy()
        )

    else:
        previous_root_array = None
        predicted_root = None

    if predicted_root is not None:
        diagnostics["predicted_root"] = (
            predicted_root.copy()
        )

    # Distance from pixels inside the mask to its boundary.
    edge_distance_map = distance_transform_edt(
        mask
    )

    # Require all four corners of a cell to be valid.
    valid_cells = (
        valid_mask[:-1, :-1]
        & valid_mask[1:, :-1]
        & valid_mask[:-1, 1:]
        & valid_mask[1:, 1:]
    )

    corners = np.stack((
        data[:-1, :-1],
        data[1:, :-1],
        data[:-1, 1:],
        data[1:, 1:],
    ))

    finite_corners = np.all(
        np.isfinite(corners.real)
        & np.isfinite(corners.imag),
        axis=0,
    )

    real_crosses_zero = (
        np.min(corners.real, axis=0) <= 0
    ) & (
        np.max(corners.real, axis=0) >= 0
    )

    imag_crosses_zero = (
        np.min(corners.imag, axis=0) <= 0
    ) & (
        np.max(corners.imag, axis=0) >= 0
    )

    candidate_cells = np.argwhere(
        valid_cells
        & finite_corners
        & real_crosses_zero
        & imag_crosses_zero
    )

    diagnostics["candidate_cell_count"] = int(
        len(candidate_cells)
    )

    if len(candidate_cells) == 0:
        return finish(no_root, "no_candidate_cells")

    fitted_candidates = []

    for cell_row, cell_col in candidate_cells:
        lower_bound = np.array(
            [cell_row, cell_col],
            dtype=float,
        )

        upper_bound = lower_bound + 1.0

        try:
            solution = least_squares(
                evaluate_field,
                x0=lower_bound + 0.5,
                bounds=(
                    lower_bound,
                    upper_bound,
                ),
                xtol=1e-12,
                ftol=1e-12,
                gtol=1e-12,
                max_nfev=100,
            )
        except (
            ValueError,
            RuntimeError,
            FloatingPointError,
        ):
            continue

        if not solution.success:
            continue

        root = np.asarray(
            solution.x,
            dtype=float,
        )

        field_at_root = evaluate_field(root)

        if (
            root.shape != (2,)
            or field_at_root.shape != (2,)
            or not np.all(np.isfinite(root))
            or not np.all(np.isfinite(field_at_root))
        ):
            continue

        absolute_residual = float(
            np.linalg.norm(field_at_root)
        )

        normalized_residual = (
            absolute_residual / field_scale
        )

        nearest_row = int(
            np.clip(
                np.rint(root[0]),
                0,
                n_rows - 1,
            )
        )

        nearest_col = int(
            np.clip(
                np.rint(root[1]),
                0,
                n_cols - 1,
            )
        )

        edge_distance = float(
            edge_distance_map[
                nearest_row,
                nearest_col,
            ]
        )

        winding, winding_measurable = (
            calculate_winding(root)
        )

        if previous_root_array is None:
            distance_from_previous = np.nan
        else:
            distance_from_previous = float(
                np.linalg.norm(
                    root - previous_root_array
                )
            )

        if predicted_root is None:
            prediction_error = np.nan
        else:
            prediction_error = float(
                np.linalg.norm(
                    root - predicted_root
                )
            )

        fitted_candidates.append({
            "root": root,
            "absolute_residual": absolute_residual,
            "normalized_residual": normalized_residual,
            "edge_distance": edge_distance,
            "winding": winding,
            "winding_measurable": winding_measurable,
            "distance_from_previous": (
                distance_from_previous
            ),
            "prediction_error": prediction_error,
            "cell": (
                int(cell_row),
                int(cell_col),
            ),
        })

    diagnostics["fitted_candidate_count"] = int(
        len(fitted_candidates)
    )

    if not fitted_candidates:
        return finish(
            no_root,
            "all_candidate_fits_failed",
        )

    # Deduplicate roots from neighboring cells.
    fitted_candidates.sort(
        key=lambda candidate:
        candidate["normalized_residual"]
    )

    unique_candidates = []

    for candidate in fitted_candidates:
        duplicate = any(
            np.linalg.norm(
                candidate["root"]
                - existing["root"]
            ) < duplicate_tolerance
            for existing in unique_candidates
        )

        if not duplicate:
            unique_candidates.append(candidate)

    valid_candidates = []

    for candidate in unique_candidates:
        residual = candidate[
            "normalized_residual"
        ]

        # Hard numerical rejection.
        if (
            not np.isfinite(residual)
            or residual
            > max_normalized_residual
        ):
            candidate["rejection"] = (
                "residual"
            )
            continue

        # A candidate on the outside-most mask pixels is unreliable.
        if (
            candidate["edge_distance"]
            < minimum_edge_distance
        ):
            candidate["rejection"] = (
                "mask_edge"
            )
            continue

        # Hard step limit.
        distance = candidate[
            "distance_from_previous"
        ]

        if (
            np.isfinite(distance)
            and distance > max_step
        ):
            candidate["rejection"] = (
                "step_too_large"
            )
            continue

        # Hard prediction-error limit, but only after two roots exist.
        prediction_error = candidate[
            "prediction_error"
        ]

        if (
            is_valid_root(previous_previous_root)
            and np.isfinite(prediction_error)
            and prediction_error
            > max_prediction_error
        ):
            candidate["rejection"] = (
                "prediction_error_too_large"
            )
            continue

        # Validate measurable winding.
        if candidate["winding_measurable"]:
            winding = candidate["winding"]

            unit_winding = (
                np.isfinite(winding)
                and abs(abs(winding) - 1.0)
                <= winding_tolerance
            )

            if not unit_winding:
                candidate["rejection"] = (
                    "not_unit_winding"
                )
                continue

            # Chirality should not reverse during tracking.
            if expected_winding in (-1, 1):
                candidate_sign = int(
                    np.sign(winding)
                )

                if candidate_sign != expected_winding:
                    candidate["rejection"] = (
                        "wrong_winding_sign"
                    )
                    continue

        candidate["rejection"] = None
        valid_candidates.append(candidate)

    diagnostics["valid_candidate_count"] = int(
        len(valid_candidates)
    )

    diagnostics["candidates"] = (
        unique_candidates
    )

    if not valid_candidates:
        rejection_reasons = {
            candidate.get(
                "rejection",
                "unknown",
            )
            for candidate in unique_candidates
        }

        reason = (
            "no_valid_candidate: "
            + ",".join(sorted(rejection_reasons))
        )

        return finish(no_root, reason)

    # Score the candidates that passed all hard tests.
    for candidate in valid_candidates:
        residual_score = min(
            candidate["normalized_residual"]
            / max_normalized_residual,
            1.0,
        )

        if np.isfinite(
            candidate["prediction_error"]
        ):
            motion_score = (
                candidate["prediction_error"]
                / max_prediction_error
            )

        elif np.isfinite(
            candidate["distance_from_previous"]
        ):
            motion_score = (
                candidate["distance_from_previous"]
                / max_step
            )

        else:
            motion_score = 0.0

        # Prefer candidates farther inside the mask.
        edge_score = 1.0 / max(
            candidate["edge_distance"],
            1.0,
        )

        # Confirmed unit winding is stronger evidence than
        # unmeasurable winding near the mask edge.
        if candidate["winding_measurable"]:
            winding_score = abs(
                abs(candidate["winding"]) - 1.0
            )
        else:
            winding_score = 0.20

        candidate["score"] = float(
            0.15 * residual_score
            + 0.60 * motion_score
            + 0.15 * edge_score
            + 0.10 * winding_score
        )

    best = min(
        valid_candidates,
        key=lambda candidate:
        candidate["score"],
    )

    diagnostics.update({
        "accepted": True,
        "reason": "accepted",
        "root": best["root"].copy(),
        "normalized_residual": best[
            "normalized_residual"
        ],
        "absolute_residual": best[
            "absolute_residual"
        ],
        "winding": best["winding"],
        "winding_measurable": best[
            "winding_measurable"
        ],
        "distance_to_edge": best[
            "edge_distance"
        ],
        "distance_from_previous": best[
            "distance_from_previous"
        ],
        "prediction_error": best[
            "prediction_error"
        ],
        "score": best["score"],
    })

    if return_diagnostics:
        return best["root"].copy(), diagnostics

    return best["root"].copy()

In [ ]:
def rotate_data(data):
    data = (
        data.isig[0].data
        + 1j * data.isig[1].data
    )
    deg = 30 # Kanskje 90 grader feil
    rot = np.exp(1j * np.deg2rad(deg))
    data = data * rot
    return np.stack((data.real, data.imag), axis=-1)

In [ ]:
def warp_data(data, transformation, shift_vec):
    data_warp = warp(
        data,
        inverse_map=AffineTransform(matrix = transformation.params),
        preserve_range=True
    )
    
    data_shift_x = shift(
        data_warp[:, :, 0],
        shift_vec
    )
    
    data_shift_y = shift(
        data_warp[:, :, 1],
        shift_vec
    )
    
    return (
        data_shift_x
        + 1j * data_shift_y
    )

In [ ]:
# def normalize_imag(data, mask):
#     def normalize(data):
#         axis = (-2, -1)
#         data = data-data[mask].min()
#         data = data/data[mask].max()
#         return data*2-1
#     real_norm = normalize(data.real)
#     imag_norm = normalize(data.imag)*1j
#     return real_norm+imag_norm

In [ ]:
s_bs_corr_dir = {}
for s_name in s_dir:
    s = s_dir[s_name]
    s_bs = s.get_direct_beam_position(method="center_of_mass")
    s_bs.compute()
    s_bs_lp = s_bs.get_linear_plane()
    s_bs_corr_dir[s_name] = s_bs-s_bs_lp

In [ ]:
fig, ax = plt.subplots()

# Get result

In [ ]:
# Generate irreversible vortex trajectories.
#
# Run this cell after:
#   1. transformations and shifts have been calculated
#   2. masks have been calculated
#   3. rotate_data and warp_data have been defined
#   4. the new find_root cell has been run

import numpy as np


# ------------------------------------------------------------
# Preparation
# ------------------------------------------------------------

image_names = sorted(
    s_dir.keys(),
    key=int,
)

number_of_images = len(image_names)
number_of_vortices = len(masks)

# result[vortex, image] = [row, column]
result = np.full(
    (
        number_of_vortices,
        number_of_images,
        2,
    ),
    np.nan,
    dtype=float,
)

root_diagnostics = [
    [
        None
        for _ in range(number_of_images)
    ]
    for _ in range(number_of_vortices)
]

# Once False, this is never set back to True.
vortex_is_active = np.ones(
    number_of_vortices,
    dtype=bool,
)

# Store the last two accepted roots for prediction.
previous_roots = [
    None
    for _ in range(number_of_vortices)
]

previous_previous_roots = [
    None
    for _ in range(number_of_vortices)
]

# Winding sign established while the vortex is safely measurable.
expected_windings = [
    None
    for _ in range(number_of_vortices)
]

# Record the first image where each vortex disappears.
disappearance_image = [
    None
    for _ in range(number_of_vortices)
]


# ------------------------------------------------------------
# Register every vector field once
# ------------------------------------------------------------

registered_fields = {}

for image_name in image_names:
    rotated_data = rotate_data(
        s_bs_corr_dir[image_name]
    )

    registered_fields[image_name] = warp_data(
        rotated_data,
        transformations[image_name],
        shifts[image_name],
    )


# ------------------------------------------------------------
# Extract irreversible trajectories
# ------------------------------------------------------------

for image_index, image_name in enumerate(
    image_names
):
    registered_data = registered_fields[
        image_name
    ]

    active_before_image = int(
        np.sum(vortex_is_active)
    )

    accepted_in_image = 0
    disappeared_in_image = 0

    for vortex_index, mask in enumerate(masks):

        # Irreversible disappearance:
        # do not search for this vortex again.
        if not vortex_is_active[vortex_index]:
            root_diagnostics[
                vortex_index
            ][image_index] = {
                "accepted": False,
                "reason": "already_disappeared",
                "root": np.array(
                    [np.nan, np.nan],
                    dtype=float,
                ),
            }

            continue

        root, diagnostic = find_root(
            registered_data,
            mask,

            previous_root=(
                previous_roots[
                    vortex_index
                ]
            ),

            previous_previous_root=(
                previous_previous_roots[
                    vortex_index
                ]
            ),

            expected_winding=(
                expected_windings[
                    vortex_index
                ]
            ),

            # Numerical root-quality limit.
            max_normalized_residual=0.05,

            # Hard trajectory limits.
            max_step=6.0,
            max_prediction_error=4.0,

            # Topological validation.
            winding_radius=2.0,
            winding_tolerance=0.25,

            # Permit roots one pixel from the digital mask edge,
            # but not outside the mask.
            minimum_edge_distance=1.0,

            duplicate_tolerance=0.25,
            return_diagnostics=True,
        )

        root_diagnostics[
            vortex_index
        ][image_index] = diagnostic

        if np.all(np.isfinite(root)):
            result[
                vortex_index,
                image_index,
            ] = root

            accepted_in_image += 1

            # Establish chirality from a measurable unit winding.
            if (
                expected_windings[
                    vortex_index
                ] is None
                and diagnostic[
                    "winding_measurable"
                ]
                and np.isfinite(
                    diagnostic["winding"]
                )
                and abs(
                    abs(
                        diagnostic["winding"]
                    ) - 1.0
                ) <= 0.25
            ):
                expected_windings[
                    vortex_index
                ] = int(
                    np.sign(
                        diagnostic["winding"]
                    )
                )

            # Shift the trajectory history.
            previous_previous_roots[
                vortex_index
            ] = previous_roots[
                vortex_index
            ]

            previous_roots[
                vortex_index
            ] = root.copy()

        else:
            # Disappearance is immediate and irreversible.
            vortex_is_active[
                vortex_index
            ] = False

            disappearance_image[
                vortex_index
            ] = image_name

            disappeared_in_image += 1

    print(
        f"Image {image_name}: "
        f"{accepted_in_image}/"
        f"{active_before_image} active roots accepted; "
        f"{disappeared_in_image} disappeared; "
        f"{np.sum(vortex_is_active)} remain active"
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

valid_roots = np.all(
    np.isfinite(result),
    axis=2,
)

print()
print("Irreversible root tracking complete")
print("-----------------------------------")
print(f"Images: {number_of_images}")
print(f"Vortices initially: {number_of_vortices}")
print(
    f"Vortices surviving through image "
    f"{image_names[-1]}: "
    f"{np.sum(vortex_is_active)}"
)
print(
    f"Vortices disappeared: "
    f"{np.sum(~vortex_is_active)}"
)
print(
    f"Total accepted roots: "
    f"{np.sum(valid_roots)}"
)

print()
print("Disappearances by image:")

for image_name in image_names:
    count = sum(
        disappeared_at == image_name
        for disappeared_at
        in disappearance_image
    )

    if count > 0:
        print(
            f"  Image {image_name}: "
            f"{count}"
        )

# Plotting for inspection

In [ ]:
from matplotlib.patches import Circle
import numpy as np
import matplotlib.pyplot as plt

# Select vortex using zero-based indexing.
vortex_idx = 16

# Keep the same ordering used when result was created:
# 002, 003, ..., 023
image_names = sorted(s_dir.keys(), key=int)

mask_center_rc = np.asarray(center[vortex_idx], dtype=float)
mask_radius = float(radii[vortex_idx])

row0, col0 = mask_center_rc

# Crop around the selected vortex.
half_width = max(14, int(np.ceil(1.8 * mask_radius)))

rmin = max(0, int(np.floor(row0 - half_width)))
rmax = min(masks[vortex_idx].shape[0],
           int(np.ceil(row0 + half_width + 1)))

cmin = max(0, int(np.floor(col0 - half_width)))
cmax = min(masks[vortex_idx].shape[1],
           int(np.ceil(col0 + half_width + 1)))

# Generate registered x-shift images only for visualization.
# The magnetic centers are read directly from result.
x_shift_images = {}

for s_name in image_names:
    vector_data = rotate_data(s_bs_corr_dir[s_name])

    registered_field = warp_data(
        vector_data,
        transformations[s_name],
        shifts[s_name]
    )

    x_shift_images[s_name] = registered_field.real

In [ ]:
n_images = len(image_names)
ncols = 6
nrows = int(np.ceil(n_images / ncols))

# Determine common grayscale limits across all displayed crops.
crop_values = []

for s_name in image_names:
    crop = x_shift_images[s_name][rmin:rmax, cmin:cmax]
    finite = crop[np.isfinite(crop)]

    if finite.size > 0:
        crop_values.append(finite)

if crop_values:
    crop_values = np.concatenate(crop_values)
    display_limit = np.nanpercentile(np.abs(crop_values), 98)

    if not np.isfinite(display_limit) or display_limit == 0:
        display_limit = 1.0
else:
    display_limit = 1.0

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(3.2 * ncols, 3.2 * nrows),
    constrained_layout=True
)

axes = np.asarray(axes).ravel()
last_im = None

for image_idx, (ax, s_name) in enumerate(zip(axes, image_names)):
    x_crop = x_shift_images[s_name][rmin:rmax, cmin:cmax]

    last_im = ax.imshow(
        x_crop,
        cmap="gray",
        origin="upper",
        vmin=-display_limit,
        vmax=display_limit,
        interpolation="nearest"
    )

    # The cyan circle shows the structural mask.
    circle = Circle(
        (col0 - cmin, row0 - rmin),
        radius=mask_radius,
        fill=False,
        color="cyan",
        linewidth=1.2
    )
    ax.add_patch(circle)

    # Use the previously calculated center.
    root_rc = result[vortex_idx][image_idx]

    valid_root = (
        root_rc is not None
        and len(root_rc) == 2
        and root_rc[0] is not None
        and root_rc[1] is not None
        and np.all(np.isfinite(root_rc))
    )

    if valid_root:
        root_row, root_col = root_rc

        # Matplotlib expects x=column and y=row.
        ax.plot(
            root_col - cmin,
            root_row - rmin,
            marker="+",
            color="red",
            markersize=13,
            markeredgewidth=2.2
        )

        ax.plot(
            root_col - cmin,
            root_row - rmin,
            marker="o",
            markerfacecolor="none",
            markeredgecolor="red",
            markersize=9,
            markeredgewidth=1.2
        )

        ax.set_title(
            f"{s_name}\n"
            f"x={root_col:.2f}, y={root_row:.2f}",
            fontsize=9
        )
    else:
        ax.set_title(
            f"{s_name}\nNO ROOT",
            fontsize=9,
            color="red"
        )

    ax.set_xticks([])
    ax.set_yticks([])

# Hide the two unused panels in the 4 x 6 grid.
for ax in axes[n_images:]:
    ax.axis("off")

fig.suptitle(
    f"Vortex {vortex_idx}: stored magnetic center on x-shift",
    fontsize=15
)

if last_im is not None:
    cbar = fig.colorbar(
        last_im,
        ax=axes[:n_images].tolist(),
        shrink=0.75,
        pad=0.015
    )
    cbar.set_label("x-shift")

plt.show()

# Create good figure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle

# Choose vortex here, using zero-based indexing.
vortex_idx = 16

image_names = sorted(s_dir, key=int)

# Use 20 of the 22 images to make a clean 4 by 5 grid.
nrows, ncols = 4, 5
plot_names = image_names[:nrows * ncols]

row0, col0 = center[vortex_idx]
radius = radii[vortex_idx]
half_width = max(14, int(np.ceil(1.8 * radius)))

shape = registered_fields[plot_names[0]].shape
rmin = max(0, int(np.floor(row0 - half_width)))
rmax = min(shape[0], int(np.ceil(row0 + half_width + 1)))
cmin = max(0, int(np.floor(col0 - half_width)))
cmax = min(shape[1], int(np.ceil(col0 + half_width + 1)))

# Magnitude of the warped complex signal.
crops = [
    registered_fields[name].real[rmin:rmax, cmin:cmax]
    for name in plot_names
]

# Shared scale makes the field evolution comparable between panels.
finite_values = np.concatenate([
    crop[np.isfinite(crop)].ravel()
    for crop in crops
])

vmin = 0
vmax = np.percentile(finite_values, 98)
cmap = sns.color_palette("vlag", as_cmap=True)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(8, 6.4),
    sharex=True,
    sharey=True,
)

for image_idx, (ax, name, crop) in enumerate(
    zip(axes.flat, plot_names, crops)
):
    ax.imshow(
        crop,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        origin="upper",
        interpolation="nearest",
    )

    # Structural vortex mask.
    ax.add_patch(Circle(
        (col0 - cmin, row0 - rmin),
        radius,
        fill=False,
        color="cyan",
        linewidth=1.0,
    ))

    # Tracked magnetic-vortex center.
    root = result[vortex_idx, image_idx]

    if np.all(np.isfinite(root)):
        row, col = root
        x, y = col - cmin, row - rmin

        ax.plot(
            x, y,
            marker="o",
            markersize=6,
            markerfacecolor="none",
            markeredgecolor="red",
            markeredgewidth=1.2,
        )
        ax.plot(
            x, y,
            marker="+",
            markersize=6,
            color="red",
            markeredgewidth=1.2,
        )

    ax.set_axis_off()

plt.subplots_adjust(
    left=0,
    right=1,
    bottom=0,
    top=1,
    wspace=0.02,
    hspace=0.02,
)

plt.show()

# Plot roots on top of color

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
from matplotlib.patches import Circle

# Choose vortex here, using zero-based indexing.
vortex_idx = 16

image_names = sorted(s_dir, key=int)
nrows, ncols = 4, 5
plot_names = image_names[:nrows * ncols]

row0, col0 = center[vortex_idx]
radius = radii[vortex_idx]
half_width = max(14, int(np.ceil(1.8 * radius)))

shape = registered_fields[plot_names[0]].shape
rmin = max(0, int(np.floor(row0 - half_width)))
rmax = min(shape[0], int(np.ceil(row0 + half_width + 1)))
cmin = max(0, int(np.floor(col0 - half_width)))
cmax = min(shape[1], int(np.ceil(col0 + half_width + 1)))

fields = [
    registered_fields[name][rmin:rmax, cmin:cmax]
    for name in plot_names
]

# One magnitude scale for all panels.
all_magnitudes = np.concatenate([
    np.abs(field[np.isfinite(field)]).ravel()
    for field in fields
])
magnitude_limit = np.percentile(all_magnitudes, 98)

def direction_rgb(field, rotation=135):
    angle = np.angle(field) + np.deg2rad(rotation)
    hue = np.mod(angle, 2 * np.pi) / (2 * np.pi)
    value = np.clip(np.abs(field) / magnitude_limit, 0, 1)

    # Keep weak regions visible without overwhelming the direction colors.
    value = 0.18 + 0.82 * np.sqrt(value)

    hsv = np.stack((
        hue,
        np.ones_like(hue),
        value,
    ), axis=-1)

    rgb = hsv_to_rgb(hsv)
    rgb[~np.isfinite(field)] = 0
    return rgb

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(8, 6.4),
    sharex=True,
    sharey=True,
)

for image_idx, (ax, field) in enumerate(
    zip(axes.flat, fields)
):
    ax.imshow(
        direction_rgb(field),
        origin="upper",
        interpolation="nearest",
    )

    # Structural vortex mask.
    ax.add_patch(Circle(
        (col0 - cmin, row0 - rmin),
        radius,
        fill=False,
        color="cyan",
        linewidth=1.0,
    ))

    # Tracked magnetic-vortex center.
    root = result[vortex_idx, image_idx]

    if np.all(np.isfinite(root)):
        row, col = root
        x, y = col - cmin, row - rmin

        # White ring with black center mark remains visible on all hues.
        ax.plot(
            x, y,
            marker="o",
            markersize=7,
            markerfacecolor="none",
            markeredgecolor="white",
            markeredgewidth=1.5,
        )
        ax.plot(
            x, y,
            marker="+",
            markersize=7,
            color="black",
            markeredgewidth=1.5,
        )

    ax.set_axis_off()

plt.subplots_adjust(
    left=0,
    right=1,
    bottom=0,
    top=1,
    wspace=0.02,
    hspace=0.02,
)

plt.show()

In [ ]:
plt.savefig("magnetic_vortex_tracking_ch.png", dpi=300, bbox_inches="tight", pad_inches=0)